In [ ]:
### Install requirements
from pathlib import Path

REPOSITORY = "https://github.com/lmingari/olot-course.git"
REQUIREMENTS = "requirements-1.txt"

if not Path(REQUIREMENTS).exists():
    !git clone -q {REPOSITORY}
    %cd olot-course
    !pip install -q -r {REQUIREMENTS}

# 1. Exploring a FALL3D output

---

FALL3D simulation results are stored in **NetCDF** files.

[NetCDF][netcdf] is a format widely used for storing multidimensional
scientific data, such as atmospheric and meteorological fields.

We will use [xarray][xarray] to open, inspect, and analyse these files.

[netcdf]: https://www.unidata.ucar.edu/software/netcdf "NetCDF"
[xarray]: https://docs.xarray.dev/ "xarray"

## Data structure

`xarray` provides two main data structures for working with scientific data:

- **DataArray**: a labelled N-dimensional array.
- **Dataset**: a collection of related DataArrays that share dimensions and coordinates.

An `xarray` object has **named dimensions and coordinates**. This makes it easier to work with scientific data such as latitude, longitude, altitude, and time.

For example, a weather forecast can be represented as a Dataset
containing several DataArrays, such as temperature and wind speed.

![Dataset and DataArray structures](https://docs.xarray.dev/en/stable/_images/dataset-diagram.png)

## &#128221; Example 1.1: loading and exploring data

In [ ]:
### Importing modules
import xarray as xr                              # multi-dimensional arrays for NetCDF data
import numpy as np                               # numerical operations on arrays
import matplotlib.pyplot as plt                  # plots and visualizations
import cartopy.crs as crs                        # coordinate systems for maps
import cartopy.feature as cfeature

In [ ]:
### Get data
from modules.helper import download_file

fname = download_file("https://saco.csic.es/s/RscfFwe7XWgDi84/download/olot_20251024.res.nc")

### Open the NetCDF file

We can open the NetCDF file with `xarray.open_dataset()`:

In [ ]:
ds = xr.open_dataset(fname)
ds

### Selecting variables and indexing

A Dataset can contain several variables. We can select one of them by name:

In [ ]:
da_times = ds["tephra_col_mass"]
da_times

This returns a **DataArray**.

We can then select a particular value along one of its dimensions using `.isel()`. For example, to select the 24-h forecast:

In [ ]:
da = da_times.isel(time=24)
da

`.isel()` uses integer positions, starting from 0.

**Summary:**

```text
Dataset (ds)
     │
     │ ["tephra_col_mass"]
     ▼
3D DataArray
     │
     │ .isel(time=24)
     ▼
2D DataArray (da)
```

## &#128221; Example 1.2: plotting on the map

In [ ]:
from modules.plotting import DataPlotter

plotter = DataPlotter(da)

In [ ]:
###
### Volcà del Montsacopa
###
vlat, vlon = 42.187193, 2.488525
###
### Add vent location
###
plotter.add_marker(vlon,vlat)

In [ ]:
###
### Add filled contours
###
plotter.contourf(levels=[0.1, 0.5, 1, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 6, 7, 8, 9, 10])

## &#128221; Example 1.3: working with probabilistic data

Forecasts of volcanic clouds are subject to uncertainties in the eruption source parameters and atmospheric conditions. **Ensemble simulations** represent these uncertainties by running the model multiple times with different input parameters.

Let's work with an ensemble simulation of an eruption at La Palma.

In [ ]:
### Get ensemble data
fname = download_file("https://saco.csic.es/s/wFpKYG5bHfTwKbi/download/tephra_col_mass.ens.nc")

In [ ]:
# Open a netCDF file in a xarray dataset
ds = xr.open_dataset(fname)
da = ds['tephra_col_mass']
da

Notice that the `DataArray` now has an additional `ens` dimension, which identifies the different ensemble members. 

We can compute the **ensemble mean** by averaging over the `ens` dimension:

In [ ]:
ensemble_mean = da.mean(dim="ens")
ensemble_mean

The **ensemble spread** measures how much the ensemble members differ from each other. A common measure of spread is the standard deviation:

In [ ]:
ensemble_spread = da.std(dim="ens")
ensemble_spread

Both operations remove the `ens` dimension, leaving a field for each forecast time and location.

In [ ]:
plotter.set_data(ensemble_mean)
plotter.contourf(levels=[1,5,10,15,20,25,30,35])

In [ ]:
plotter.set_data(ensemble_spread)
plotter.contourf(levels=[1,5,10,15,20,25,30,35])

## &#9998; Exercise: Exceedance probabilities

Given an ensemble of column mass forecasts:

- Compute the **exceedance probability** for a threshold of $10~\mathrm{g/m^2}$.
- Plot contour lines for probabilities of **2%, 25%, 50%, 75%, and 98%**.

**Hint:** The percentage of ensemble members exceeding a threshold can be computed as:

```python
100 * (da > threshold).mean(dim="ens")
```

<details>
<summary> Show solution</summary>

```python
threshold = 10
probability = 100 * (da > threshold).mean(dim="ens")

plotter.set_data(probability)
plotter.contour(levels=[2, 25, 50, 75, 98])
```

</details>